In [ ]:
import tensorflow as tf                     # importiert TensorFlow
import numpy as np                          # importiert NumPy für Arrays

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()  # lädt MNIST (60k Training, 10k Test)

x_train = (x_train / 255.0).astype(np.float32)[..., None]  # skaliert Pixel auf 0..1 und hängt Kanalachse an (28,28,1)
x_test  = (x_test  / 255.0).astype(np.float32)[..., None]  # dasselbe für die Testdaten

model = tf.keras.Sequential([                               # baut das Modell Schicht für Schicht
    tf.keras.layers.Input(shape=(28, 28, 1)),               # Eingang: ein Graustufenbild 28x28
    tf.keras.layers.Conv2D(8, 3, activation="relu"),        # 8 Filter 3x3 -> 26x26x8
    tf.keras.layers.MaxPooling2D(),                         # halbiert die Größe -> 13x13x8
    tf.keras.layers.Conv2D(16, 3, activation="relu"),       # 16 Filter 3x3 -> 11x11x16
    tf.keras.layers.MaxPooling2D(),                         # halbiert -> 5x5x16
    tf.keras.layers.Flatten(),                              # macht daraus einen Vektor mit 400 Werten
    tf.keras.layers.Dense(10, activation="softmax"),        # 10 Ausgänge = Wahrscheinlichkeit pro Ziffer
])

model.compile(optimizer="adam",                             # Optimierer
              loss="sparse_categorical_crossentropy",       # Verlustfunktion für Klassenlabels 0-9
              metrics=["accuracy"])                         # misst die Genauigkeit
model.summary()                                             # zeigt Schichten und Parameterzahl

model.fit(x_train, y_train, epochs=5,                       # trainiert 5 Durchläufe
          batch_size=64, validation_split=0.1)              # 10 % der Trainingsdaten zur Validierung

loss, acc = model.evaluate(x_test, y_test, verbose=0)       # testet auf allen 10.000 Testbildern
print(f"Keras float32 test accuracy: {acc:.4f}")            # gibt die Genauigkeit aus

model.save("mnist_cnn.keras")                               # speichert das trainierte Modell

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 8)      │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 16)     │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │         4,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,258 (20.54 KB)

 Trainable params: 5,258 (20.54 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 15s 17ms/step - accuracy: 0.8918 - loss: 0.3737 - val_accuracy: 0.9652 - val_loss: 0.1225
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.9651 - loss: 0.1167 - val_accuracy: 0.9775 - val_loss: 0.0818
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.9739 - loss: 0.0858 - val_accuracy: 0.9782 - val_loss: 0.0733
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.9779 - loss: 0.0726 - val_accuracy: 0.9822 - val_loss: 0.0646
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.9809 - loss: 0.0632 - val_accuracy: 0.9830 - val_loss: 0.0619
Keras float32 test accuracy: 0.9820


In [ ]:
model.export("saved_model")                                   # exportiert das Modell als SavedModel (stabilster Weg für den Konverter)

# --- float32 ---
conv = tf.lite.TFLiteConverter.from_saved_model("saved_model")  # erstellt den Konverter
tflite_f32 = conv.convert()                                   # wandelt ohne Quantisierung um
open("mnist_f32.tflite", "wb").write(tflite_f32)              # speichert die Datei

# --- int8 ---
def representative_data():                                    # Kalibrierdaten, wie in deinem Lab
    for i in range(200):                                      # 200 Trainingsbilder reichen
        yield [x_train[i:i+1]]                                # gibt jeweils ein Bild (1,28,28,1) zurück

conv = tf.lite.TFLiteConverter.from_saved_model("saved_model")  # neuer Konverter
conv.optimizations = [tf.lite.Optimize.DEFAULT]               # schaltet Quantisierung ein
conv.representative_dataset = representative_data             # übergibt die Kalibrierdaten
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]  # erzwingt reine int8-Operationen
conv.inference_input_type = tf.int8                           # Eingang wird int8
conv.inference_output_type = tf.int8                          # Ausgang wird int8
tflite_i8 = conv.convert()                                    # wandelt um
open("mnist_int8.tflite", "wb").write(tflite_i8)              # speichert die Datei

print("float32 model size:", len(tflite_f32), "bytes")        # Größe float32
print("int8 model size:   ", len(tflite_i8), "bytes")         # Größe int8

Saved artifact at 'saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  132227851810000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132227851810960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132227851810576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132227851811344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132227851811152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132227851810768: TensorSpec(shape=(), dtype=tf.resource, name=None)
float32 model size: 24992 bytes
int8 model size:    10552 bytes


In [ ]:
def evaluate(path):                                        # testet ein .tflite-Modell auf allen 10.000 Testbildern
    interp = tf.lite.Interpreter(model_path=path)          # lädt das Modell in den TFLite-Interpreter
    interp.allocate_tensors()                              # reserviert Speicher für die Tensoren
    inp = interp.get_input_details()[0]                    # Infos zum Eingang (Index, Typ, Quantisierung)
    out = interp.get_output_details()[0]                   # Infos zum Ausgang
    correct = 0                                            # Zähler für richtige Vorhersagen
    for i in range(len(x_test)):                           # geht jedes Testbild durch
        x = x_test[i:i+1]                                  # nimmt ein Bild (1,28,28,1)
        if inp["dtype"] == np.int8:                        # nur beim int8-Modell:
            scale, zp = inp["quantization"]                # holt Skalierungsfaktor und Nullpunkt
            x = np.clip(np.round(x / scale + zp), -128, 127).astype(np.int8)  # rechnet float -> int8 um
        interp.set_tensor(inp["index"], x)                 # legt das Bild an den Eingang
        interp.invoke()                                    # führt die Inferenz aus
        pred = np.argmax(interp.get_tensor(out["index"]))  # nimmt die Klasse mit dem höchsten Wert
        correct += int(pred == y_test[i])                  # zählt, ob sie stimmt
    return correct / len(x_test)                           # gibt die Genauigkeit zurück

print("TFLite float32 accuracy:", evaluate("mnist_f32.tflite"))   # Genauigkeit float32
print("TFLite int8 accuracy:   ", evaluate("mnist_int8.tflite"))  # Genauigkeit int8

i8 = tf.lite.Interpreter(model_path="mnist_int8.tflite")   # lädt das int8-Modell nochmal
print("input quantization:", i8.get_input_details()[0]["quantization"])  # Skalierung + Nullpunkt, brauchen wir später auf dem ESP32

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


TFLite float32 accuracy: 0.982
TFLite int8 accuracy:    0.9816
input quantization: (0.003921568859368563, -128)


In [ ]:
print("Keras float32 accuracy:", model.evaluate(x_test, y_test, verbose=0)[1])  # misst das aktuelle Keras-Modell erneut

Keras float32 accuracy: 0.9819999933242798


In [ ]:
def c_array(data, name, ctype="unsigned char", align=False):   # wandelt Bytes/Zahlen in C-Code um
    vals = ", ".join(str(int(v)) for v in data)                 # macht aus den Werten eine Komma-Liste
    a = "alignas(16) " if align else ""                         # TFLite Micro braucht das Modell 16-Byte-ausgerichtet
    return f"{a}const {ctype} {name}[] = {{{vals}}};\nconst unsigned int {name}_len = {len(data)};\n\n"  # erzeugt Array + Länge

with open("model_data.h", "w") as f:                            # öffnet die Header-Datei für die Modelle
    f.write("#pragma once\n\n")                                 # verhindert doppeltes Einbinden
    f.write(c_array(tflite_f32, "model_f32", align=True))       # schreibt das float32-Modell
    f.write(c_array(tflite_i8, "model_int8", align=True))       # schreibt das int8-Modell

N = 100                                                         # Anzahl Testbilder für den ESP32
(_, _), (x_raw, y_raw) = tf.keras.datasets.mnist.load_data()    # lädt die Rohbilder (0-255) erneut
with open("test_data.h", "w") as f:                             # öffnet die Header-Datei für die Testdaten
    f.write("#pragma once\n\n")                                 # verhindert doppeltes Einbinden
    f.write(f"#define NUM_TEST_IMAGES {N}\n\n")                 # legt die Bildanzahl als Konstante fest
    f.write(c_array(x_raw[:N].flatten(), "test_images"))        # 100 Bilder x 784 Pixel als uint8
    f.write(c_array(y_raw[:N], "test_labels"))                  # die 100 richtigen Antworten

from google.colab import files                                  # Colab-Werkzeug zum Herunterladen
files.download("model_data.h")                                  # lädt die Modell-Datei herunter
files.download("test_data.h")                                   # lädt die Testdaten-Datei herunter

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>